# GeoZones — Filtered zones

Builds `zones_filtered.geojson` by combining:
- `zones_global.geojson` (6 top-level zones, from `geozones-global.ipynb`)
- FR territorial zones downloaded from data.gouv.fr

Output: `zones_filtered.geojson`

In [1]:
import subprocess
from collections import defaultdict

try:
    import ijson
except ImportError:
    subprocess.check_call(["uv", "add", "ijson"])
    import ijson

DOWNLOAD_URL   = "https://demo.data.gouv.fr/api/1/datasets/r/7971e15c-3255-4724-b690-77171bc6d9ad"
NEW_ZONES_FILE = "./export-geozones.json"
GLOBAL_FILE    = "./zones_global.geojson"
OUT_FILE       = "./zones_filtered.geojson"

print("ijson", ijson.__version__)

ijson 3.5.0


In [2]:
import urllib.request, os

if not os.path.exists(NEW_ZONES_FILE):
    print(f"Downloading {DOWNLOAD_URL} ...")
    urllib.request.urlretrieve(DOWNLOAD_URL, NEW_ZONES_FILE)
    print(f"Done. ({os.path.getsize(NEW_ZONES_FILE)/1024/1024:.1f} MB)")
else:
    print(f"Already exists: {NEW_ZONES_FILE}  ({os.path.getsize(NEW_ZONES_FILE)/1024/1024:.1f} MB)")

Done. (81.9 MB)


In [3]:
import json, os

counters = defaultdict(int)
first = True

with open(OUT_FILE, "w") as dst:
    dst.write('{"type":"FeatureCollection","features":[')

    # Part 1: global zones (world, EU, fr, fr:metro, fr:drom, fr:dromcom)
    with open(GLOBAL_FILE) as f:
        global_zones = json.load(f)
    for feature in global_zones["features"]:
        if not first:
            dst.write(",")
        dst.write(json.dumps(feature, ensure_ascii=False))
        first = False
        counters[feature.get("id", "")] += 1

    # Part 2: FR territorial zones from export-geozones.json
    seen_ids = set()
    with open(NEW_ZONES_FILE, "rb") as src:
        for item in ijson.items(src, "item", use_float=True):
            if not item.get("geom"):
                continue
            zone_id = item["_id"]
            if zone_id in seen_ids:
                continue
            seen_ids.add(zone_id)

            feature = {
                "type": "Feature",
                "id": zone_id,
                "geometry": item["geom"],
                "properties": {
                    "level": item.get("level", ""),
                    "nom":   item.get("nom", ""),
                    "uri":   item.get("uri", ""),
                },
            }
            dst.write(",")
            dst.write(json.dumps(feature, ensure_ascii=False))
            counters[item.get("level", "unknown")] += 1

    dst.write("]}")

print(f"Done. Output: {OUT_FILE}  ({os.path.getsize(OUT_FILE)/1024/1024:.1f} MB)")
print("\nFeature counts:")
for key, cnt in sorted(counters.items()):
    print(f"  {key:<35} {cnt:>6}")

Done. Output: ./zones_filtered.geojson  (50.4 MB)

Feature counts:
  country-group:ue                         1
  country-group:world                      1
  country-subset:fr:drom                   1
  country-subset:fr:dromcom                1
  country-subset:fr:metro                  1
  country:fr                               1
  fr:commune                           36952
  fr:departement                         109
  fr:epci                               1255
  fr:region                               18
